# Тема 8. Рассуждение и планирование: цикл TAO и одиночный ReAct-агент

**Модуль III · Пример 1 из 5**

Сквозной домен модуля — поддержка интернет-магазина «ШопБот». В этой тетради — базовая архитектура: **одиночный агент** по циклу **TAO** (Thought → Action → Observation), стратегия **ReAct**.

- **Thought** — модель рассуждает и планирует шаг.
- **Action** — вызывает инструмент (или формирует финальный ответ).
- **Observation** — результат инструмента возвращается в контекст.
- Цикл повторяется, пока задача не решена. Механизм **stop-and-parse**: без остановки на вызове модель «выдумала» бы результат инструмента.

Одиночный ReAct-агент — точка отсчёта (baseline) для сравнения с мультиагентными системами в следующих тетрадях.

### Доступ к модели
Параметры доступа берутся из **переменных окружения** (ключ в код не вставляется):
`LLM_API_KEY`, `LLM_BASE_URL` (адрес OpenAI-совместимого эндпоинта), `LLM_MODEL`.

In [1]:
import os, json, time
from openai import OpenAI

BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:8000/v1")
API_KEY  = os.environ.get("LLM_API_KEY", "")
MODEL    = os.environ.get("LLM_MODEL", "qwen/qwen3.7-max")

client = OpenAI(api_key=API_KEY or "EMPTY", base_url=BASE_URL)
print("Эндпоинт:", BASE_URL, "| модель:", MODEL)

Эндпоинт: <LLM_BASE_URL> | модель: qwen/qwen3.7-max


## Домен и инструменты

Инструменты вынесены в общий модуль `mas_domain.py` — один и тот же набор используют и одиночный агент, и мультиагентные системы (для корректной сравнимости). Пять инструментов: поиск товаров, заказ, политика, расчёт возврата, статус доставки.

In [2]:
import mas_domain as dom

print("Инструменты:", [t["function"]["name"] for t in dom.TOOLS_SPEC])
# Прямой вызов инструмента (без модели):
print("search_products('hub') ->", dom.run_tool("search_products", {"query": "hub"}))

Инструменты: ['search_products', 'get_order', 'get_policy', 'calc_refund', 'check_shipping']
search_products('hub') -> [{'sku': 'P-300', 'name': 'USB-C Hub', 'price': 2490, 'stock': 15}]


## Реализация ReAct-агента с явной трассой TAO

Петля «модель ⇄ инструменты»: на каждом шаге печатаем **Thought** (рассуждение модели, если оно есть), **Action** (вызов инструмента) и **Observation** (результат). `parallel_tool_calls=False` форсирует один вызов за шаг — так цикл TAO виден пошагово. `max_steps` — простая защита от зацикливания.

In [3]:
def react_agent(user_query: str, model: str = MODEL, max_steps: int = 8, verbose: bool = True):
    messages = [
        {"role": "system", "content":
            "Ты — ассистент поддержки интернет-магазина. Работай по циклу "
            "Thought->Action->Observation: рассуждай, вызывай инструменты, используй их "
            "результаты. Каталог на английском — в query передавай английские слова. "
            "Заверши, только когда покрыты ВСЕ части запроса пользователя."},
        {"role": "user", "content": user_query},
    ]
    n_calls = 0
    for step in range(max_steps):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, tools=dom.TOOLS_SPEC,
                parallel_tool_calls=False, max_tokens=700)
        except TypeError:
            resp = client.chat.completions.create(
                model=model, messages=messages, tools=dom.TOOLS_SPEC, max_tokens=700)
        n_calls += 1
        msg = resp.choices[0].message
        messages.append(msg.model_dump())
        if msg.content and verbose:
            print(f"[{step+1}] Thought: {msg.content.strip()[:160]}")
        if not msg.tool_calls:
            return {"answer": msg.content, "steps": step + 1, "llm_calls": n_calls}
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = dom.run_tool(tc.function.name, args)
            if verbose:
                print(f"    Action: {tc.function.name}({args})")
                print(f"    Observation: {json.dumps(result, ensure_ascii=False)[:160]}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    return {"answer": "Достигнут лимит шагов.", "steps": max_steps, "llm_calls": n_calls}

print("react_agent определён.")

react_agent определён.


## Тест на задачах возрастающей сложности

Проверяем поведение на трёх уровнях (как в теме 8): одношаговая → многошаговая с явной последовательностью → многошаговая с описанием только цели.

In [4]:
print("=== Уровень 1: одношаговая ===")
r1 = react_agent("Найди в каталоге USB-C hub")
print("\nОтвет:", r1["answer"][:300], "\n(шагов:", r1["steps"], "| вызовов модели:", r1["llm_calls"], ")")

=== Уровень 1: одношаговая ===


    Action: search_products({'query': 'USB-C hub'})
    Observation: [{"sku": "P-300", "name": "USB-C Hub", "price": 2490, "stock": 15}]


[2] Thought: Нашёл в каталоге:

**USB-C Hub** (артикул P-300)
- 💰 Цена: **2 490 ₽**
- 📦 В наличии: **15 шт.**

Если хотите оформить заказ или узнать подробнее — напишите!

Ответ: Нашёл в каталоге:

**USB-C Hub** (артикул P-300)
- 💰 Цена: **2 490 ₽**
- 📦 В наличии: **15 шт.**

Если хотите оформить заказ или узнать подробнее — напишите! 
(шагов: 2 | вызовов модели: 2 )


In [5]:
print("=== Уровень 2: многошаговая с явной последовательностью ===")
r2 = react_agent("Проверь заказ ORD-1002, а затем узнай статус его доставки")
print("\nОтвет:", r2["answer"][:400], "\n(шагов:", r2["steps"], "| вызовов модели:", r2["llm_calls"], ")")

=== Уровень 2: многошаговая с явной последовательностью ===


[1] Thought: Сейчас проверю информацию о заказе и статус доставки.
    Action: get_order({'order_id': 'ORD-1002'})
    Observation: {"order_id": "ORD-1002", "items": [{"sku": "P-300", "qty": 2}], "total": 4980, "status": "shipped", "date": "2026-07-15"}
    Action: check_shipping({'order_id': 'ORD-1002'})
    Observation: {"order_id": "ORD-1002", "carrier": "CDEK", "state": "in_transit", "eta_days": 2}


[2] Thought: Вот информация по заказу **ORD-1002**:

## Детали заказа
- **Товар:** P-300 (2 шт.)
- **Сумма:** 4 980 ₽
- **Статус заказа:** Отправлен (shipped)
- **Дата заказ

Ответ: Вот информация по заказу **ORD-1002**:

## Детали заказа
- **Товар:** P-300 (2 шт.)
- **Сумма:** 4 980 ₽
- **Статус заказа:** Отправлен (shipped)
- **Дата заказа:** 15 июля 2026

## Статус доставки
- **Перевозчик:** CDEK
- **Состояние:** В пути (in_transit)
- **Ожидаемый срок доставки:** 2 дня

Заказ уже отправлен и находится в пути. Ориентировочная доставка — через 2 дня. Если есть ещё вопросы, о 
(шагов: 2 | вызовов модели: 2 )


In [6]:
print("=== Уровень 3: многодоменная, описана только цель ===")
r3 = react_agent(
    "Я хочу вернуть заказ ORD-1001 и заодно понять, за сколько дней приедет новый USB-C хаб, "
    "если закажу сегодня. Посчитай сумму возврата и учти политику возврата.")
print("\nОтвет:", r3["answer"][:600], "\n(шагов:", r3["steps"], "| вызовов модели:", r3["llm_calls"], ")")

=== Уровень 3: многодоменная, описана только цель ===


    Action: calc_refund({'order_id': 'ORD-1001'})
    Observation: {"refundable": true, "amount": 4990, "reason": "в пределах политики возврата (14 дней)"}
    Action: search_products({'query': 'USB-C hub'})
    Observation: [{"sku": "P-300", "name": "USB-C Hub", "price": 2490, "stock": 15}]
    Action: get_policy({'topic': 'refund'})
    Observation: {"topic": "refund", "text": "Возврат в течение 14 дней с момента доставки, товар без следов использования. Комиссия за возврат — 0 ₽."}
    Action: get_policy({'topic': 'shipping'})
    Observation: {"topic": "shipping", "text": "Стандартная доставка 3–5 дней; экспресс 1–2 дня (+300 ₽)."}


[2] Thought: Вот полная информация по обоим вашим вопросам:

---

## 💰 Возврат заказа ORD-1001

- **Статус:** ✅ Возврат возможен
- **Сумма возврата:** **4 990 ₽**
- **Причин

Ответ: Вот полная информация по обоим вашим вопросам:

---

## 💰 Возврат заказа ORD-1001

- **Статус:** ✅ Возврат возможен
- **Сумма возврата:** **4 990 ₽**
- **Причина:** Заказ находится в пределах 14-дневного срока с момента доставки.

**Условия возврата:**
- Товар должен быть без следов использования.
- Комиссия за возврат — **0 ₽** (вы получите полную сумму).

---

## 🚚 Доставка нового USB-C хаба

| Вариант | Срок | Стоимость доставки |
|---|---|---|
| Стандартная | **3–5 дней** | Бесплатно |
| Экспресс | **1–2 дня** | +300 ₽ |

В наличии — **USB-C Hub (P-300)** за **2 490 ₽** (15 шт. на складе). 
(шагов: 2 | вызовов модели: 2 )


## Итоги

- **TAO** — базовый цикл любого агента: рассуждение → действие → наблюдение, с остановкой на вызове инструмента (**stop-and-parse**).
- **ReAct** превосходит «цепочку рассуждений» на задачах с инструментами, потому что корректируется через Observation.
- На простых и явных запросах одиночный агент справляется. На **многодоменной** задаче (уровень 3) один агент со множеством обязанностей склонен **терять часть доменов** или путать контексты — это мотивирует переход к мультиагентным системам.

**Дальше:** Тема 9 — типы взаимодействия между агентами (коллаборация, дебаты, конкуренция).